# pypgo.energy API Demo

This tutorial demonstrates every public energy type in the `pypgo.energy`
module: `LinearEnergy`, `QuadraticEnergy`, `VertexAttachment`, and
`EnergySet`.

**Audience:** users building energy-based models (FEM, IPC, optimization)
with pypgo.

**Prerequisites:** NumPy basics and familiarity with the concept of
potential energy.

**Learning goals:**

1. Construct `LinearEnergy` and `QuadraticEnergy` from NumPy arrays.
2. Pin vertices with `VertexAttachment`.
3. Combine energies with `EnergySet` and adjust weights.
4. Evaluate energy, gradient, and Hessian through the unified API.
5. Understand `state_kind` and `zero_state` conventions.


## Outline

1. Import and setup
2. LinearEnergy — b^T x
3. QuadraticEnergy — 1/2 x^T A x + b^T x
4. VertexAttachment — soft pin constraints
5. EnergySet — weighted sum of energies
6. state_kind and zero_state conventions
7. Composite Hessian and max_step
8. Lifetime and ownership
9. Exercise


In [170]:
import numpy as np
import pypgo.energy as pe


## 1. Import and setup

All energy types live in `pypgo.energy`.  They share the same evaluation
methods — `value(x)`, `gradient(x)`, `hessian(x)`, `max_step(x, dx)`,
`zero_state()` — and the same properties `num_dofs`, `dofs`, `state_kind`.


In [171]:
# Quick sanity: what's available?
for name in sorted(dir(pe)):
    if not name.startswith("_"):
        print(name)


EnergySet
LinearEnergy
QuadraticEnergy
SparseMatrix
VertexAttachment
annotations
np


## 2. LinearEnergy — b^T x

The simplest energy term.  The coefficient vector `b` is **copied** into
C++ owned storage, so you can mutate or delete the Python array after
construction.


In [172]:
b = np.array([1.0, -2.0, 3.0, 4.0], dtype=np.float64)
lin = pe.LinearEnergy(b)

print("num_dofs:", lin.num_dofs)
print("state_kind:", lin.state_kind)
print(repr(lin))


num_dofs: 4
state_kind: generic
LinearEnergy(4 DOFs)


In [173]:
x = np.array([1.0, 0.5, -1.0, 0.0], dtype=np.float64)

e = lin.value(x)
g = lin.gradient(x)
H = lin.hessian(x)

print(f"value  = {e:.6f}  (expected b·x = 1*1 + -2*0.5 + 3*-1 + 4*0 = {1*1 + (-2)*0.5 + 3*(-1) + 4*0})")
print(f"gradient = {g}  (always equals b)")
print("hessian (dense):")
print(H.to_dense())
print(f"(nnz={H.nnz} — linear energy has no curvature)")


value  = -3.000000  (expected b·x = 1*1 + -2*0.5 + 3*-1 + 4*0 = -3.0)
gradient = [ 1. -2.  3.  4.]  (always equals b)
hessian (dense):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
(nnz=0 — linear energy has no curvature)


In [174]:
# Ownership: delete b — energy still works
b_original = np.array([10.0, 20.0, 30.0], dtype=np.float64)
owned = pe.LinearEnergy(b_original)
del b_original
x = np.ones(3, dtype=np.float64)
print("value after b deleted:", owned.value(x))


value after b deleted: 60.0


## 3. QuadraticEnergy — 1/2 x^T A x + b^T x

The `A` matrix can be provided as a **5-tuple** `(rows, cols,
row_indices, col_indices, values)` or as a `PySparseMatrix`.  `b` is
optional.


In [175]:
# 5-tuple COO input: A = diag(2, 3, 4)
q = pe.QuadraticEnergy(
    (3, 3,                              # shape
     [0, 1, 2],                         # row_indices
     [0, 1, 2],                         # col_indices
     np.array([2.0, 3.0, 4.0], dtype=np.float64)),  # values
)
print("num_dofs:", q.num_dofs)
print("state_kind:", q.state_kind)
print(repr(q))


num_dofs: 3
state_kind: generic
QuadraticEnergy(3 DOFs)


In [176]:
x = np.ones(3, dtype=np.float64)

# value = 1/2 * (2·1² + 3·1² + 4·1²) = 4.5
print("value:", q.value(x))

# gradient = A @ x = [2, 3, 4]
print("gradient:", q.gradient(x))

# Hessian = A = diag(2, 3, 4)
H = q.hessian(x)
rows, cols, vals = H.to_coo()
print("hessian coo values:", vals)


value: 4.5
gradient: [2. 3. 4.]
hessian coo values: [2. 3. 4.]


### Non-diagonal Hessian

A matrix with off-diagonal entries produces a non-trivial Hessian
pattern visible in `to_dense()`.


In [177]:
# A = [[2, 1, 0],
#      [1, 3, 1],
#      [0, 1, 4]]
A_coo = (3, 3,
         [0, 0, 1, 1, 1, 2, 2],     # row_indices
         [0, 1, 0, 1, 2, 1, 2],     # col_indices
         [2.0, 1.0, 1.0, 3.0, 1.0, 1.0, 4.0])
q_nd = pe.QuadraticEnergy(A_coo)

H = q_nd.hessian(np.ones(3))
print("Hessian (dense):")
print(H.to_dense())
print(f"\nshape={H.shape}, nnz={H.nnz}")


Hessian (dense):
[[2. 1. 0.]
 [1. 3. 1.]
 [0. 1. 4.]]

shape=(3, 3), nnz=7


In [178]:
# With linear term: 1/2 x^T A x + b^T x
qb = pe.QuadraticEnergy(
    (2, 2,
     [0, 1],
     [0, 1],
     np.array([2.0, 0.0], dtype=np.float64)),
    b=np.array([1.0, 3.0], dtype=np.float64),
)
x = np.array([2.0, 0.0], dtype=np.float64)
# value = 0.5 * 2 * 4 + 1*2 + 3*0 = 4 + 2 = 6.0
print("value with b:", qb.value(x))
g = qb.gradient(x)
# gradient = A@x + b = [4, 0] + [1, 3] = [5, 3]
print("gradient with b:", g)


value with b: 6.0
gradient with b: [5. 3.]


### Dense ndarray input

You can also pass a 2-D NumPy array directly — zeros are automatically
dropped during COO conversion.


In [179]:
# Dense ndarray input — zeros are dropped automatically
A_dense = np.diag([2.0, 3.0, 4.0])
q_dense = pe.QuadraticEnergy(A_dense)

x = np.ones(3, dtype=np.float64)
print("value from dense:", q_dense.value(x))
print("gradient from dense:", q_dense.gradient(x))

# COO tuple still works too
q_coo = pe.QuadraticEnergy(
    (2, 2,
     [0, 1],             # row_indices (list of int)
     [0, 1],             # col_indices (list of int)
     [10.0, 20.0]),      # values (plain list)
)
print("value from coo:", q_coo.value(np.array([1.0, 0.0])))


value from dense: 4.5
gradient from dense: [2. 3. 4.]
value from coo: 5.0


## 4. VertexAttachment — soft pin constraints

`VertexAttachment` pins selected vertices to target positions with a
quadratic penalty: `coeff · ||u_i − target_i||²`.  It is a **displacement**
energy (`state_kind == "displacement"`) — it assumes `x` is a displacement
from the rest configuration.


In [180]:
# Pin vertex 0 to (0,0,0) and vertex 1 to (1,0,0) in a 3-vertex system
n_dofs = 9  # 3 vertices × 3 dof
koff = (n_dofs, n_dofs,
        list(range(n_dofs)),
        list(range(n_dofs)),
        [1.0] * n_dofs)

pin = pe.VertexAttachment(
    koff=koff,
    vertex_indices=np.array([0, 1], dtype=np.int64),
    target_positions=np.array(
        [0.0, 0.0, 0.0,   # target for vertex 0
         1.0, 0.0, 0.0],  # target for vertex 1
        dtype=np.float64,
    ),
    coeff=1000.0,
)
print("num_dofs:", pin.num_dofs)
print("state_kind:", pin.state_kind)


num_dofs: 9
state_kind: displacement


In [181]:
# At zero displacement, vertices 0 and 1 are at rest (0,0,0) and (1,0,0)?
# Wait — if is_displacement=True (default), then absolute position =
# rest + displacement.  Our rest_positions are all-zero here, so the
# targets are absolute positions in this simplified setup.
u = np.zeros(n_dofs, dtype=np.float64)
print("energy at zero disp:", pin.value(u))

# Move vertex 0 away from its target
u_pert = u.copy()
u_pert[0] = 0.5   # displace vertex 0 x by 0.5
print("energy after perturbation:", pin.value(u_pert))
print("energy increase:", pin.value(u_pert) - pin.value(u))


energy at zero disp: 500.0
energy after perturbation: 625.0
energy increase: 125.0


In [182]:
# Gradient points toward the targets
g = pin.gradient(u)
# For vertex 0 pinned to (0,0,0) at coeff 1000:
# g[0:3] = coeff * (rest + u - target) = coeff * (0 + 0 - 0) = 0
# For vertex 1 pinned to (1,0,0):
# g[3:6] = coeff * (rest + u - target) = 1000 * (0 + 0 - 1, 0, 0) = (-1000, 0, 0)
print("gradient at pinned vertices:\n", g.reshape(-1, 3))


gradient at pinned vertices:
 [[    0.     0.     0.]
 [-1000.     0.     0.]
 [    0.     0.     0.]]


## 5. EnergySet — weighted sum of energies

`EnergySet` composes multiple energy terms into a single energy:
`total = Σ weight_i · energy_i`.  Construction is one-shot — no
separate `add`/`init` step.


In [183]:
# Create individual energies
A_mat = (3, 3,
         [0, 1, 2],
         [0, 1, 2],
         np.array([100.0, 200.0, 300.0], dtype=np.float64))
elastic = pe.QuadraticEnergy(A_mat)

force = pe.LinearEnergy(np.array([0.0, -9.81, 0.0], dtype=np.float64))

# Combine with weights
total = pe.EnergySet([
    (elastic,   1.0),   # elastic energy × 1
    (force,    -1.0),   # − b^T u  (external work)
])
print(repr(total))


EnergySet(2 terms, 3 DOFs, state_kind='generic')


In [184]:
u = np.array([0.01, -0.02, 0.0], dtype=np.float64)

print("total value:     ", total.value(u))
print("elastic only:    ", elastic.value(u))
print("force only:      ", force.value(u))


total value:      -0.1512
elastic only:     0.045
force only:       0.1962


In [185]:
# Adjust weights dynamically
total.set_weight(1, 0.0)  # disable external force
print("after disabling force:", total.value(u))

total.set_weight(1, -1.0)  # re-enable
print("after re-enabling:    ", total.value(u))


after disabling force: 0.045
after re-enabling:     -0.1512


## 6. state_kind and zero_state conventions

`state_kind` is `"displacement"` for FEM/contact/attachment energies and
`"generic"` for linear/quadratic energies.  `EnergySet` composes them:
all-children-displacement → `"displacement"`, otherwise `"generic"`.


In [186]:
n = 3
koff = (n, n, list(range(n)), list(range(n)), [1.0] * n)
pin = pe.VertexAttachment(
    koff=koff,
    vertex_indices=np.array([0], dtype=np.int64),
    target_positions=np.zeros(3, dtype=np.float64),
)
lin = pe.LinearEnergy(np.ones(n, dtype=np.float64))

print("pin state_kind:  ", pin.state_kind)
print("lin state_kind:  ", lin.state_kind)

es_pin = pe.EnergySet([(pin, 1.0)])
es_mix = pe.EnergySet([(pin, 1.0), (lin, 1.0)])

print("pin-only set:    ", es_pin.state_kind)
print("mixed set:       ", es_mix.state_kind)


pin state_kind:   displacement
lin state_kind:   generic
pin-only set:     displacement
mixed set:        generic


In [187]:
# zero_state() always returns a zero array of the right dtype
for e in [pin, lin, elastic, total]:
    z = e.zero_state()
    print(f"{type(e).__name__:20s}  zero_state: shape={z.shape}, dtype={z.dtype}")


VertexAttachment      zero_state: shape=(3,), dtype=float64
LinearEnergy          zero_state: shape=(3,), dtype=float64
QuadraticEnergy       zero_state: shape=(3,), dtype=float64
EnergySet             zero_state: shape=(3,), dtype=float64


## 7. Composite Hessian and max_step

The Hessian of an `EnergySet` is assembled automatically from its
children.  `max_step` delegates to each child's barrier (if any) and
returns the most restrictive limit.


In [188]:
# Hessian of combined energy — returns a SparseMatrix
H = total.hessian(u)
print(f"total Hessian: {H.shape[0]}×{H.shape[1]}, nnz={H.nnz}")

# COO export
rows, cols, vals = H.to_coo()
print("coo values:", vals)

# Dense export
H_dense = H.to_dense()
print("\ndense Hessian:\n", H_dense)


total Hessian: 3×3, nnz=3
coo values: [100. 200. 300.]

dense Hessian:
 [[100.   0.   0.]
 [  0. 200.   0.]
 [  0.   0. 300.]]


In [189]:
# max_step returns a MaxStepResult with alpha + clamp info
dx = np.ones_like(u)
ms = total.max_step(u, dx)
print(f"alpha:             {ms.alpha}")
print(f"material_alpha:    {ms.material_alpha}")
print(f"contact_alpha:     {ms.contact_alpha}")
print(f"material_clamped:  {ms.material_clamped}")
print(f"contact_clamped:   {ms.contact_clamped}")


alpha:             1.0
material_alpha:    1.0
contact_alpha:     1.0
material_clamped:  False
contact_clamped:   False


## 8. Lifetime and ownership

Every energy type **owns** its data in C++.  You can delete the Python
wrapper of a child energy after adding it to an `EnergySet` — the set
keeps the C++ object alive through `shared_ptr`.


In [190]:
import gc

q = pe.QuadraticEnergy(
    (2, 2, [0, 1], [0, 1], np.array([5.0, 5.0], dtype=np.float64)),
)
es = pe.EnergySet([(q, 1.0)])
del q
gc.collect()

x = np.ones(2, dtype=np.float64)
print("value after child deletion:", es.value(x))


value after child deletion: 5.0


## 9. Exercise

Build a simple 2-DOF system:
1. A quadratic energy with `A = diag(10, 20)`.
2. A linear energy with `b = [1, 0]`.
3. Combine them in an `EnergySet` with weights `(1.0, -1.0)`.
4. Evaluate at `x = [1, 1]` and verify the gradient manually.
5. Temporarily set the linear term weight to 0 and re-evaluate.


In [191]:
# Your solution here
q = pe.QuadraticEnergy(
    (2, 2, [0, 1], [0, 1], np.array([10.0, 20.0], dtype=np.float64)),
)
lin = pe.LinearEnergy(np.array([1.0, 0.0], dtype=np.float64))
es = pe.EnergySet([(q, 1.0), (lin, -1.0)])

x = np.array([1.0, 1.0], dtype=np.float64)

# value = 0.5*(10*1 + 20*1) - 1*1 = 15 - 1 = 14
print("value:", es.value(x))
# gradient: A@x = [10, 20]; -b = [-1, 0]; total = [9, 20]
print("gradient:", es.gradient(x))

es.set_weight(1, 0.0)
print("value without linear term:", es.value(x))
print("gradient without linear term:", es.gradient(x))


value: 14.0
gradient: [ 9. 20.]
value without linear term: 15.0
gradient without linear term: [10. 20.]


## Pitfalls and extensions

**Pitfall:** all energies in an `EnergySet` must have the same
`num_dofs`.  A `ValueError` is raised if they differ.

**Pitfall:** `VertexAttachment` `is_displacement=True` (default) means
the state `x` is interpreted as a displacement from rest.  Set
`is_displacement=False` if your state is absolute positions.

**Extension:** for FEM deformation energies, use the `pypgo.fem` module
(once available).  The resulting `DeformationEnergy` is a
`"displacement"`-kind energy that can be combined in the same
`EnergySet`.
